In [1]:
from pycocotools.coco import COCO
ann_file = "../../data/matching/annotations/instances_default.json"
coco = COCO(ann_file)

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


In [4]:
# load all annotations with category_id = 2 and valid knot id
annotations = [ann for ann in coco.anns.values() if ann['category_id'] == 2 and ann['attributes']['id'] != -1]
# extract id, attributes["id"] and segmentation from each annotation
annotations_parsed = []
for ann in annotations:
    ann_id = ann['id']
    image_name = coco.loadImgs(ann['image_id'])[0]['file_name']
    knot_id = ann['attributes']['id']
    segmentation = ann['segmentation']
    annotations_parsed.append({
        'ann_id': int(ann_id),
        'image_name': image_name,
        'knot_id': int(knot_id),
        'segmentation': segmentation
    })
    

In [6]:
len(annotations_parsed)

125

In [2]:
from mltools.matching.utils import parse_coco_annotations

ann_file = "../../data/matching/annotations/instances_default.json"
parsed_annotations = parse_coco_annotations(ann_file)

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!


In [3]:
len(parsed_annotations)

125

In [6]:
len(set([ann['image_name'] for ann in parsed_annotations]))

23

In [70]:
from mltools.matching.utils import parse_coco_annotations
from mltools.matching.volume import VoxelGrid
from pathlib import Path
import numpy as np
import cv2
from matplotlib import pyplot as plt

def construct_voxel_grid(annotations: list[dict], projections_dir: str, voxel_size: float = 0.1) -> VoxelGrid:
    image_ids = set([Path(ann['image_name']).stem for ann in annotations.values()])
    projections = {}
    for image_id in image_ids:
        projection_file = Path(projections_dir) / f"{image_id}.npy"
        if projection_file.exists():
            projections[image_id] = np.load(projection_file)
        else:
            print(f"Warning: Projection file {projection_file} does not exist.")
            projections[image_id] = None

    vg = VoxelGrid(voxel_size=voxel_size)
    for ann_id, ann in annotations.items():
        image_id = Path(ann['image_name']).stem
        projection = projections[image_id]
        if projection is None:
            continue
        pos = projection[..., :3]
        valid_mask = projection[..., 3] > 0.5
        h, w = projection.shape[:2]
        segmentation = ann['segmentation']
        # Create a mask from the segmentation that is in normalized coordinates (0-1) and then scale it to the image size
        mask = np.zeros((h, w), dtype=np.uint8)
        for seg in segmentation:
            seg = np.array(seg).reshape((-1, 2))
            seg[:, 0] *= w
            seg[:, 1] *= h
            poly = seg.astype(np.int32)
            cv2.fillPoly(mask, [poly], 1)
        # print(f"Segmentation mask for ann_id {ann['ann_id']} has {np.sum(mask)} pixels.")
        combined_mask = valid_mask & (mask > 0)
        points = pos[combined_mask]
        # print(ann['ann_id'], ann['image_name'], points.shape, ann['image_id'])

        vg.add_volume(points, volume_id=ann_id, group_id=int(image_id))
    return vg

In [71]:
ann_file = "../../data/matching/annotations/instances_default.json"
projections_dir = "../../data/matching/resized_1024_3d"
voxel_size = 0.005
annotations = parse_coco_annotations(ann_file)
vg = construct_voxel_grid(annotations, projections_dir, voxel_size)

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!


In [76]:
structures = vg.find_structures(merge_threshold = 0.3)

In [77]:
len(structures)

80

In [78]:
def confusion_matrix(annotations: dict[dict], structures: list[dict]) -> np.ndarray:
    # Create a mapping from annotation_id to structure_id
    ann_to_structure = {}
    for str_id, structure in enumerate(structures):
        for ann_id in structure:
            ann_to_structure[ann_id] = str_id

    # iterate over all pairs of annotations and count true positives, false positives, true negatives, and false negatives
    tp = 0
    fp = 0
    tn = 0
    fn = 0
    anns = []
    for ann_id, ann in annotations.items():
        ann['ann_id'] = ann_id
        anns.append(ann)

    for i in range(len(anns)):
        for j in range(i + 1, len(anns)):
            ann_i = anns[i]
            ann_j = anns[j]
            same_knot = ann_i['knot_id'] == ann_j['knot_id']
            same_structure = ann_to_structure.get(ann_i['ann_id']) == ann_to_structure.get(ann_j['ann_id'])
            if same_knot and same_structure:
                tp += 1
            elif not same_knot and same_structure:
                fp += 1
            elif same_knot and not same_structure:
                fn += 1
            else:
                tn += 1

    return np.array([[tp, fp], [fn, tn]])


In [79]:
cm = confusion_matrix(annotations, structures)
print(cm)

[[  40   32]
 [  15 7663]]


In [33]:
precision = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0
recall = cm[0, 0] / (cm[0, 0] + cm[1, 0]) if (cm[0, 0] + cm[1, 0]) > 0 else 0
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

Precision: 0.1014, Recall: 0.5273
